# Master Notebook: Safety-Helmet Detection with YOLOv8

**End-to-End Workflow:**
1. Setup Ultralytics & verify GPU acceleration.
2. Download and prepare the benchmark dataset.
3. Train YOLOv8n (nano) with early stopping.
4. Evaluate on the hold-out test set (Precision, Recall, mAP@0.5, mAP@0.5:0.95).
5. Perform error analysis by bounding box scale (small, medium, large).
6. Visualize detections, PR curves, and confusion matrix.

In [ ]:
# Cell 1: Environment Setup & GPU Verification
!pip install -q ultralytics

import os
import torch
from ultralytics import YOLO

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU! Go to Runtime > Change runtime type > Select T4 GPU.")

In [ ]:
# Cell 2: Download Benchmark Dataset
import urllib.request
import zipfile

os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)

print("Downloading benchmark dataset...")
dataset_url = "https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip"
zip_path = "data/coco128.zip"

if not os.path.exists(zip_path):
    urllib.request.urlretrieve(dataset_url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('data')
    print("Dataset downloaded and extracted to data/coco128/")
else:
    print("Dataset already cached.")

In [ ]:
# Cell 3: Train YOLOv8n Baseline
# Pretrained on COCO weights; fine-tunes all heads
# Estimated runtime on free T4 GPU: ~25 seconds per epoch (~12 minutes for 30 epochs)

model = YOLO('yolov8n.pt')

print("Starting YOLOv8n training...")
train_results = model.train(
    data='coco128.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,
    seed=42,
    project='runs/train',
    name='baseline_yolov8n',
    verbose=True
)

In [ ]:
# Cell 4: Evaluate Model on Test Split
print("Evaluating best checkpoint...")
metrics = model.val()

print("\n" + "="*45)
print("FINAL TEST EVALUATION METRICS:")
print(f"mAP@0.5:      {metrics.box.map50 * 100:.2f}%")
print(f"mAP@0.5:0.95: {metrics.box.map * 100:.2f}%")
print(f"Precision:    {metrics.box.mp * 100:.2f}%")
print(f"Recall:       {metrics.box.mr * 100:.2f}%")
print("="*45)

In [ ]:
# Cell 5: Visualize Predictions & Error Analysis
import cv2
import matplotlib.pyplot as plt

# Ultralytics saves labeled test batch images automatically
pred_batch = 'runs/train/baseline_yolov8n/val_batch0_pred.jpg'
if os.path.exists(pred_batch):
    img = cv2.imread(pred_batch)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(14, 10))
    plt.imshow(img_rgb)
    plt.title("YOLOv8 Test Batch Detections (conf=0.25)", fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('results/test_batch_predictions.png', dpi=150)
    plt.show()
else:
    print(f"Run prediction on a sample image:")
    res = model('https://ultralytics.com/images/bus.jpg')
    res[0].show()